# External Native-Footprint SIF Inference

Apply the trained 4 km Sentinel-2 U-Net to centred 4 km predictor chips from the two unseen MGRS tiles. The model produces a 20 m SIF map for each chip; its prediction is averaged over the native OCO-2 footprint using the saved fractional mask and compared with observed `target_modis_sif`.

## 1. Imports And Paths

Attach two Kaggle datasets: one containing the prepared inference chips and one containing the trained `.pt` checkpoint. Only the two paths below need editing.

In [ ]:
from __future__ import annotations

from collections import OrderedDict
import json
import math
from pathlib import Path
import random

import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, Sampler

alt.data_transformers.disable_max_rows()

# Edit these after attaching the two Kaggle datasets.
CHIP_DIR = Path(
    '/kaggle/input/CHANGE_ME/external_2tiles_native_footprints_4km_20m'
)
CHECKPOINT_PATH = Path(
    '/kaggle/input/CHANGE_ME/sentinel2_spatial_aggregate_4km_unet.pt'
)

METADATA_PATH = CHIP_DIR / 'chip_metadata.csv'
OUTPUT_DIR = Path('/kaggle/working/sentinel2_external_native_footprint_inference')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
NUM_WORKERS = 2
SHARD_CACHE_SIZE = 8
USE_AMP = True
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = bool(USE_AMP and DEVICE.type == 'cuda')

print('device:', DEVICE)
print('AMP enabled:', AMP_ENABLED)
print('chip directory:', CHIP_DIR)
print('checkpoint:', CHECKPOINT_PATH)

## 2. Load Checkpoint And Rebuild The Model

The checkpoint supplies the exact channel order, training normalization statistics, target scaling, architecture width and optional validation calibration.

In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(CHECKPOINT_PATH)

try:
    checkpoint = torch.load(
        CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
except TypeError:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')

required_checkpoint_keys = [
    'model_state_dict', 'model_class', 'channel_names', 'base_channels',
    'channel_mean', 'channel_std', 'target_mean', 'target_std',
    'normalize_target', 'calibration',
]
missing_checkpoint_keys = [
    key for key in required_checkpoint_keys if key not in checkpoint
]
if missing_checkpoint_keys:
    raise KeyError(f'Checkpoint is missing keys: {missing_checkpoint_keys}')
if checkpoint['model_class'] != 'ThreeLevelUNet':
    raise ValueError(f"Unexpected model class: {checkpoint['model_class']}")

channel_names = [str(value) for value in checkpoint['channel_names']]
channel_mean = np.asarray(checkpoint['channel_mean'], dtype=np.float32)
channel_std = np.asarray(checkpoint['channel_std'], dtype=np.float32)
target_mean = float(checkpoint['target_mean'])
target_std = float(checkpoint['target_std'])
normalize_target = bool(checkpoint['normalize_target'])
calibration = dict(checkpoint['calibration'])

if len(channel_names) != 19:
    raise ValueError(f'Expected 19 channels, found {len(channel_names)}')
if channel_mean.shape != (len(channel_names),):
    raise ValueError('Checkpoint channel_mean shape is invalid')
if channel_std.shape != (len(channel_names),):
    raise ValueError('Checkpoint channel_std shape is invalid')
if not np.isfinite(channel_mean).all():
    raise ValueError('Checkpoint channel means contain non-finite values')
if not np.isfinite(channel_std).all() or (channel_std <= 0).any():
    raise ValueError('Checkpoint channel standard deviations are invalid')
if normalize_target and (not np.isfinite(target_std) or target_std <= 0):
    raise ValueError(f'Invalid checkpoint target standard deviation: {target_std}')

display(pd.DataFrame({
    'channel_index': np.arange(len(channel_names)),
    'channel': channel_names,
    'training_mean': channel_mean,
    'training_std': channel_std,
}))
print('target mean:', target_mean)
print('target std:', target_std)
print('calibration:', calibration)

In [ ]:
def group_count(channels: int) -> int:
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return groups
    return 1


class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        groups = group_count(out_channels)
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ThreeLevelUNet(nn.Module):
    def __init__(self, in_channels: int, base_channels: int = 16):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base_channels * 2, base_channels * 4)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base_channels * 4, base_channels * 8)
        self.up3 = nn.ConvTranspose2d(
            base_channels * 8, base_channels * 4, 2, stride=2
        )
        self.dec3 = ConvBlock(base_channels * 8, base_channels * 4)
        self.up2 = nn.ConvTranspose2d(
            base_channels * 4, base_channels * 2, 2, stride=2
        )
        self.dec2 = ConvBlock(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(
            base_channels * 2, base_channels, 2, stride=2
        )
        self.dec1 = ConvBlock(base_channels * 2, base_channels)
        self.out = nn.Conv2d(base_channels, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        bottleneck = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(bottleneck), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)


model = ThreeLevelUNet(
    in_channels=len(channel_names),
    base_channels=int(checkpoint['base_channels']),
).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.eval()

print('best training epoch:', checkpoint.get('best_epoch'))
print('best validation RMSE:', checkpoint.get('best_validation_rmse'))
print('trainable parameters:', sum(p.numel() for p in model.parameters()))

## 3. Index And Validate Inference Shards

Every shard is checked against the checkpoint channel order and merged one-to-one with `chip_metadata.csv`.

In [ ]:
def decode_strings(values: np.ndarray) -> list[str]:
    output = []
    for value in values:
        if isinstance(value, bytes):
            output.append(value.decode('utf-8'))
        else:
            output.append(str(value))
    return output


if not METADATA_PATH.exists():
    raise FileNotFoundError(METADATA_PATH)

metadata = pd.read_csv(METADATA_PATH, parse_dates=['Delta_Date', 'par_date'])
shard_files = sorted(CHIP_DIR.glob('chips_*.npz'))
if not shard_files:
    raise FileNotFoundError(f'No chips_*.npz files found in {CHIP_DIR}')

index_rows = []
for shard_id, shard_path in enumerate(shard_files):
    with np.load(shard_path, allow_pickle=False) as shard:
        current_channels = decode_strings(shard['channel_names'])
        sample_ids = decode_strings(shard['sample_id'])
        targets = shard['observed_sif']
        if current_channels != channel_names:
            raise ValueError(
                f'Channel mismatch between checkpoint and {shard_path.name}'
            )
        if shard['X'].shape[1:] != (len(channel_names), 200, 200):
            raise ValueError(f'Unexpected X shape in {shard_path.name}')
        if shard['footprint_mask'].shape[1:] != (200, 200):
            raise ValueError(f'Unexpected mask shape in {shard_path.name}')
        if len(sample_ids) != len(targets):
            raise ValueError(f'Shard array lengths differ in {shard_path.name}')

    for local_index, sample_id in enumerate(sample_ids):
        index_rows.append({
            'sample_id': sample_id,
            'shard_id': shard_id,
            'shard_path': str(shard_path),
            'local_index': local_index,
        })

shard_index = pd.DataFrame(index_rows)
if shard_index['sample_id'].duplicated().any():
    raise ValueError('Duplicate sample IDs found across NPZ shards')
if metadata['sample_id'].duplicated().any():
    raise ValueError('Duplicate sample IDs found in chip metadata')

samples = metadata.merge(
    shard_index, on='sample_id', how='inner', validate='one_to_one'
)
if len(samples) != len(metadata) or len(samples) != len(shard_index):
    raise ValueError(
        f'Metadata/shard mismatch: metadata={len(metadata)}, '
        f'shards={len(shard_index)}, merged={len(samples)}'
    )

samples = samples.sort_values(
    ['shard_id', 'local_index']
).reset_index(drop=True)
print('metadata rows:', len(metadata))
print('NPZ samples:', len(shard_index))
print('shards:', len(shard_files))
display(samples.groupby('mgrs_tile_t').size().rename('n').to_frame())
display(samples.head())

## 4. Dataset And DataLoader

Stored float16 arrays are cast to float32. Predictors use checkpoint statistics, NaNs become zero only after normalization, and every footprint mask is renormalized to sum to one.

In [ ]:
class NpzShardCache:
    def __init__(self, max_size: int):
        self.max_size = max(1, int(max_size))
        self.cache: OrderedDict[str, dict[str, np.ndarray]] = OrderedDict()

    def get(self, path: str) -> dict[str, np.ndarray]:
        if path in self.cache:
            self.cache.move_to_end(path)
            return self.cache[path]
        with np.load(path, allow_pickle=False) as shard:
            loaded = {
                'X': shard['X'].copy(),
                'footprint_mask': shard['footprint_mask'].copy(),
                'observed_sif': shard['observed_sif'].copy(),
            }
        self.cache[path] = loaded
        self.cache.move_to_end(path)
        while len(self.cache) > self.max_size:
            self.cache.popitem(last=False)
        return loaded


class ExternalFootprintDataset(Dataset):
    def __init__(self, table: pd.DataFrame, cache_size: int = 8):
        self.table = table.reset_index(drop=True).copy()
        self.cache = NpzShardCache(cache_size)

    def __len__(self) -> int:
        return len(self.table)

    def __getitem__(self, index: int):
        row = self.table.iloc[index]
        shard = self.cache.get(str(row['shard_path']))
        local_index = int(row['local_index'])

        x = shard['X'][local_index].astype(np.float32, copy=True)
        mask = shard['footprint_mask'][local_index].astype(
            np.float32, copy=True
        )
        observed = np.float32(shard['observed_sif'][local_index])

        x = (x - channel_mean[:, None, None]) / channel_std[:, None, None]
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        mask = np.nan_to_num(mask, nan=0.0, posinf=0.0, neginf=0.0)
        mask_sum = float(mask.sum(dtype=np.float64))
        if not np.isfinite(mask_sum) or mask_sum <= 0:
            raise ValueError(f'Invalid mask at dataset index {index}')
        mask /= mask_sum

        if not np.isfinite(observed):
            raise ValueError(f'Non-finite observed SIF at index {index}')

        return (
            torch.from_numpy(x),
            torch.from_numpy(mask),
            torch.tensor(observed, dtype=torch.float32),
            torch.tensor(index, dtype=torch.long),
        )


class ShardBatchSampler(Sampler[list[int]]):
    def __init__(self, table: pd.DataFrame, batch_size: int):
        self.batch_size = int(batch_size)
        self.groups = [
            np.asarray(indices, dtype=np.int64)
            for indices in table.groupby(
                'shard_path', sort=False
            ).indices.values()
        ]

    def __iter__(self):
        for indices in self.groups:
            for start in range(0, len(indices), self.batch_size):
                yield indices[start:start + self.batch_size].tolist()

    def __len__(self) -> int:
        return sum(
            math.ceil(len(indices) / self.batch_size)
            for indices in self.groups
        )


def seed_worker(worker_id: int) -> None:
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


dataset = ExternalFootprintDataset(samples, cache_size=SHARD_CACHE_SIZE)
batch_sampler = ShardBatchSampler(samples, BATCH_SIZE)
loader_kwargs = {
    'num_workers': NUM_WORKERS,
    'pin_memory': DEVICE.type == 'cuda',
    'worker_init_fn': seed_worker,
}
if NUM_WORKERS > 0:
    loader_kwargs.update({'persistent_workers': True, 'prefetch_factor': 2})
loader = DataLoader(
    dataset, batch_sampler=batch_sampler, **loader_kwargs
)

print('samples:', len(dataset))
print('inference batches:', len(loader))

## 5. External Footprint Inference

In [ ]:
def denormalize_prediction(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    if normalize_target:
        return values * target_std + target_mean
    return values


def apply_calibration(values: np.ndarray) -> np.ndarray:
    return (
        float(calibration['intercept'])
        + float(calibration['slope']) * np.asarray(values, dtype=np.float64)
    )


predicted_normalized = np.full(len(dataset), np.nan, dtype=np.float64)
observed_sif = np.full(len(dataset), np.nan, dtype=np.float64)

model.eval()
with torch.inference_mode():
    for batch_number, (x, mask, observed, sample_indices) in enumerate(
        loader, start=1
    ):
        x = x.to(DEVICE, non_blocking=True)
        mask_gpu = mask.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED,
        ):
            predicted_map = model(x)

        footprint_prediction = (
            predicted_map[:, 0].float() * mask_gpu.float()
        ).sum(dim=(-2, -1))

        positions = sample_indices.numpy()
        predicted_normalized[positions] = footprint_prediction.cpu().numpy()
        observed_sif[positions] = observed.numpy()

        if batch_number % 50 == 0 or batch_number == len(loader):
            print(f'Inference batch {batch_number:,} / {len(loader):,}')

if not np.isfinite(predicted_normalized).all():
    raise ValueError('Some samples did not receive finite predictions')
if not np.isfinite(observed_sif).all():
    raise ValueError('Some samples have non-finite observed SIF')

predictions = samples.copy()
predictions['observed_sif'] = observed_sif
predictions['predicted_sif_raw'] = denormalize_prediction(
    predicted_normalized
)
predictions['predicted_sif_calibrated'] = apply_calibration(
    predictions['predicted_sif_raw'].to_numpy()
)
# Raw predictions remain the primary external-validation result.
predictions['predicted_sif'] = predictions['predicted_sif_raw']
predictions['residual_raw'] = (
    predictions['predicted_sif_raw'] - predictions['observed_sif']
)
predictions['residual_calibrated'] = (
    predictions['predicted_sif_calibrated'] - predictions['observed_sif']
)
predictions['residual'] = predictions['residual_raw']

display(predictions[[
    'sample_id', 'mgrs_tile_t', 'Delta_Date', 'measurement_mode',
    'Quality_Flag', 'date_align', 'observed_sif',
    'predicted_sif_raw', 'predicted_sif_calibrated', 'residual_raw',
]].head())

## 6. Overall Metrics

Raw checkpoint predictions are primary. Validation-derived linear calibration is retained as a secondary sensitivity result.

In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if y_true.size == 0:
        return {
            'n': 0, 'rmse': np.nan, 'mae': np.nan, 'bias': np.nan,
            'r2': np.nan, 'intercept': np.nan, 'slope': np.nan,
            'pearson_r': np.nan,
        }

    residual = y_pred - y_true
    ss_res = float(np.sum(residual ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    r2 = np.nan if ss_tot <= 0 else 1.0 - ss_res / ss_tot
    if y_true.size >= 2 and np.std(y_true) > 0:
        design = np.column_stack([np.ones(y_true.size), y_true])
        intercept, slope = np.linalg.lstsq(design, y_pred, rcond=None)[0]
        pearson_r = (
            float(np.corrcoef(y_true, y_pred)[0, 1])
            if np.std(y_pred) > 0 else np.nan
        )
    else:
        intercept, slope, pearson_r = np.nan, np.nan, np.nan

    return {
        'n': int(y_true.size),
        'rmse': float(np.sqrt(np.mean(residual ** 2))),
        'mae': float(np.mean(np.abs(residual))),
        'bias': float(np.mean(residual)),
        'r2': float(r2),
        'intercept': float(intercept),
        'slope': float(slope),
        'pearson_r': float(pearson_r),
    }


overall_raw = regression_metrics(
    predictions['observed_sif'], predictions['predicted_sif_raw']
)
overall_calibrated = regression_metrics(
    predictions['observed_sif'], predictions['predicted_sif_calibrated']
)
print('External native-footprint metrics, raw:')
print(overall_raw)
print('External native-footprint metrics, calibrated:')
print(overall_calibrated)

## 7. Predicted Versus Observed

Point colour represents local two-dimensional count density. Both axes use the same limits and tick positions.

In [ ]:
def density_values(x: np.ndarray, y: np.ndarray, bins: int = 80) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    histogram, x_edges, y_edges = np.histogram2d(x, y, bins=bins)
    x_index = np.clip(np.searchsorted(x_edges, x, side='right') - 1, 0, bins - 1)
    y_index = np.clip(np.searchsorted(y_edges, y, side='right') - 1, 0, bins - 1)
    return histogram[x_index, y_index]


def prediction_chart(
    table: pd.DataFrame, prediction_column: str, title_label: str
) -> alt.Chart:
    plot_table = table[[
        'observed_sif', prediction_column, 'mgrs_tile_t'
    ]].dropna().copy()
    plot_table = plot_table.rename(columns={prediction_column: 'predicted_sif'})
    plot_table['density'] = density_values(
        plot_table['observed_sif'], plot_table['predicted_sif']
    )
    metrics = regression_metrics(
        plot_table['observed_sif'], plot_table['predicted_sif']
    )

    tick_step = 0.25
    minimum = min(plot_table['observed_sif'].min(), plot_table['predicted_sif'].min())
    maximum = max(plot_table['observed_sif'].max(), plot_table['predicted_sif'].max())
    lower = math.floor(minimum / tick_step) * tick_step
    upper = math.ceil(maximum / tick_step) * tick_step
    if upper <= lower:
        upper = lower + tick_step
    ticks = np.arange(lower, upper + tick_step * 0.5, tick_step).tolist()

    axis = alt.Axis(values=ticks, format='.2f', grid=True)
    points = alt.Chart(plot_table).mark_circle(
        size=14, opacity=0.72
    ).encode(
        x=alt.X(
            'observed_sif:Q', title='Observed OCO-2 footprint SIF',
            scale=alt.Scale(domain=[lower, upper]), axis=axis,
        ),
        y=alt.Y(
            'predicted_sif:Q', title='Predicted footprint-mean SIF',
            scale=alt.Scale(domain=[lower, upper]), axis=axis,
        ),
        color=alt.Color(
            'density:Q', title='Density',
            scale=alt.Scale(type='log', scheme='turbo'),
        ),
        tooltip=[
            alt.Tooltip('observed_sif:Q', format='.4f'),
            alt.Tooltip('predicted_sif:Q', format='.4f'),
            'mgrs_tile_t:N', alt.Tooltip('density:Q', format='.0f'),
        ],
    )

    identity_table = pd.DataFrame({
        'x': [lower, upper], 'y': [lower, upper]
    })
    identity = alt.Chart(identity_table).mark_line(
        color='black', strokeDash=[6, 4]
    ).encode(x='x:Q', y='y:Q')

    regression_table = pd.DataFrame({
        'x': [lower, upper],
        'y': [
            metrics['intercept'] + metrics['slope'] * lower,
            metrics['intercept'] + metrics['slope'] * upper,
        ],
    })
    regression = alt.Chart(regression_table).mark_line(
        color='#d62728', strokeWidth=2
    ).encode(x='x:Q', y='y:Q')

    return (points + identity + regression).properties(
        width=560, height=560,
        title=(
            f'{title_label}: RMSE={metrics["rmse"]:.4f}, '
            f'R2={metrics["r2"]:.3f}, slope={metrics["slope"]:.3f}'
        ),
    )


raw_chart = prediction_chart(predictions, 'predicted_sif_raw', 'Raw')
calibrated_chart = prediction_chart(
    predictions, 'predicted_sif_calibrated', 'Calibrated'
)
display(raw_chart)
display(calibrated_chart)
raw_chart.save(OUTPUT_DIR / 'external_predicted_vs_observed_raw.html')
calibrated_chart.save(
    OUTPUT_DIR / 'external_predicted_vs_observed_calibrated.html'
)

## 8. Metrics By Tile, Month, Mode, Alignment And Quality

SIF-bin metrics diagnose regression toward the mean. Other groups assess geographic, seasonal, retrieval-mode and temporal-alignment transfer.

In [ ]:
SIF_BIN_EDGES = [-np.inf, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0, 1.25, np.inf]
SIF_BIN_LABELS = [
    '(-inf,-0.25)', '[-0.25,0)', '[0,0.25)', '[0.25,0.5)',
    '[0.5,0.75)', '[0.75,1)', '[1,1.25)', '[1.25,inf)',
]
predictions['sif_bin'] = pd.cut(
    predictions['observed_sif'],
    bins=SIF_BIN_EDGES, labels=SIF_BIN_LABELS, right=False,
)


def group_metric_rows(table: pd.DataFrame, column: str) -> list[dict]:
    rows = []
    for value, group in table.groupby(column, observed=False, dropna=False):
        raw = regression_metrics(group['observed_sif'], group['predicted_sif_raw'])
        calibrated = regression_metrics(
            group['observed_sif'], group['predicted_sif_calibrated']
        )
        rows.append({
            'group_variable': column,
            'group_value': str(value),
            'n': int(len(group)),
            'observed_min': float(group['observed_sif'].min()) if len(group) else np.nan,
            'observed_max': float(group['observed_sif'].max()) if len(group) else np.nan,
            'observed_mean': float(group['observed_sif'].mean()) if len(group) else np.nan,
            'raw_predicted_mean': float(group['predicted_sif_raw'].mean()) if len(group) else np.nan,
            **{f'raw_{key}': value for key, value in raw.items() if key != 'n'},
            'calibrated_predicted_mean': (
                float(group['predicted_sif_calibrated'].mean()) if len(group) else np.nan
            ),
            **{
                f'calibrated_{key}': value
                for key, value in calibrated.items() if key != 'n'
            },
        })
    return rows


group_columns = [
    'sif_bin', 'mgrs_tile_t', 'sif_month', 'measurement_mode',
    'date_align', 'Quality_Flag', 'sif_year', 'hzs', 'state',
]
group_rows = []
for column in group_columns:
    if column in predictions.columns:
        group_rows.extend(group_metric_rows(predictions, column))

group_metrics = pd.DataFrame(group_rows)
for column in group_columns:
    current = group_metrics[group_metrics['group_variable'] == column]
    if len(current):
        print(column)
        display(current.reset_index(drop=True))

## 9. Residual Diagnostics

In [ ]:
residual_chart = alt.Chart(predictions).mark_bar(color='#2c7fb8').encode(
    x=alt.X(
        'residual_raw:Q', bin=alt.Bin(maxbins=70),
        title='Raw predicted - observed SIF',
    ),
    y=alt.Y('count():Q', title='Footprints'),
).properties(width=700, height=350, title='External footprint residuals')
display(residual_chart)
residual_chart.save(OUTPUT_DIR / 'external_residual_histogram.html')

tile_summary = group_metrics[
    group_metrics['group_variable'] == 'mgrs_tile_t'
].copy()
tile_chart = alt.Chart(tile_summary).mark_bar(color='#238b45').encode(
    x=alt.X('group_value:N', title='External MGRS tile'),
    y=alt.Y('raw_rmse:Q', title='Raw RMSE'),
    tooltip=['group_value:N', 'n:Q', alt.Tooltip('raw_rmse:Q', format='.4f'),
             alt.Tooltip('raw_r2:Q', format='.3f')],
).properties(width=420, height=320, title='External RMSE by tile')
display(tile_chart)
tile_chart.save(OUTPUT_DIR / 'external_tile_rmse.html')

## 10. Example 20 m Prediction Map And Footprint Mask

In [ ]:
EXAMPLE_INDEX = 0
x_example, mask_example, observed_example, _ = dataset[EXAMPLE_INDEX]
row_example = dataset.table.iloc[EXAMPLE_INDEX]

with np.load(row_example['shard_path'], allow_pickle=False) as shard:
    raw_x_example = shard['X'][int(row_example['local_index'])].astype(np.float32)

with torch.inference_mode():
    with torch.autocast(
        device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED
    ):
        map_normalized = model(x_example[None].to(DEVICE))[0, 0]

map_normalized = map_normalized.float().cpu().numpy()
map_raw = denormalize_prediction(map_normalized)
mask_array = mask_example.numpy()
predicted_example = float((map_raw * mask_array).sum())
observed_example = float(observed_example)
ndvi_index = channel_names.index('ndvi')

figure, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
image0 = axes[0].imshow(map_raw, cmap='viridis')
axes[0].set_title('Predicted 20 m SIF map')
figure.colorbar(image0, ax=axes[0], fraction=0.046)
image1 = axes[1].imshow(raw_x_example[ndvi_index], cmap='RdYlGn', vmin=-1, vmax=1)
axes[1].set_title('Raw NDVI')
figure.colorbar(image1, ax=axes[1], fraction=0.046)
image2 = axes[2].imshow(mask_array, cmap='magma')
axes[2].set_title('Normalized footprint mask')
figure.colorbar(image2, ax=axes[2], fraction=0.046)
axes[3].imshow(map_raw, cmap='viridis', alpha=0.45)
axes[3].imshow(
    np.ma.masked_where(mask_array <= 0, mask_array), cmap='Reds', alpha=0.85
)
axes[3].set_title(
    f'obs={observed_example:.4f}, pred={predicted_example:.4f}'
)
for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])
plt.show()

## 11. Save External Validation Results

In [ ]:
predictions_path = OUTPUT_DIR / 'external_footprint_predictions.csv'
group_metrics_path = OUTPUT_DIR / 'external_group_metrics.csv'
metrics_path = OUTPUT_DIR / 'external_metrics.json'

predictions.to_csv(predictions_path, index=False)
group_metrics.to_csv(group_metrics_path, index=False)
metrics_payload = {
    'checkpoint_path': str(CHECKPOINT_PATH),
    'chip_directory': str(CHIP_DIR),
    'primary_prediction': 'raw',
    'n_footprints': int(len(predictions)),
    'n_dates': int(predictions['Delta_Date'].nunique()),
    'n_products': int(predictions['product_path'].nunique()),
    'tiles': sorted(predictions['mgrs_tile_t'].astype(str).unique().tolist()),
    'raw': overall_raw,
    'calibrated': overall_calibrated,
    'training_validation_calibration': calibration,
}
with metrics_path.open('w', encoding='utf-8') as file:
    json.dump(metrics_payload, file, indent=2)

print('Saved:')
for path in [predictions_path, group_metrics_path, metrics_path]:
    print(' -', path)